# Pandas for Materials Science: Tensile Testing Data Analysis

## 1- Introduction & Setup

**Why Pandas for Materials Science?** <br>

Pandas is the industry-standard tool for data manipulation and analysis in Python. In materials science, we often work with:
- Large datasets from testing equipment (MTS, Instron, etc.)
- Multiple test replicates requiring statistical analysis
- Time-series data from sensors
- Need for reproducible data processing workflows


## 2 -  File Management with the os Module

Before we dive into data analysis, we need to organize our workspace. 

### The Folder Structure We'll Use

```
MASE6401/
├── data/
│   ├── MTS_metals/     # Excel files from MTS tensile tests
│   └── MTS_SG/         # Text-based data files
├── notebooks/          # Jupyter notebooks (this file lives here)
└── results/            # Processed data and figures
```

In [ ]:
# Import necessary libraries
import os
import pandas as pd                # pd is the standard nomenclature for pandas
import numpy as np
import matplotlib.pyplot as plt

# If "module not found" error run:
#!pip install pandas

### 2.1 Navigating the File System


The current working directory (CWD) is the folder your Python script is currently running from. It's the default location for reading and writing files

In [ ]:
# Get the current working directory
current_dir = os.getcwd()                                                   # os.getcwd() -> returns a string representing the current working directory.
print(f"Current working directory: {current_dir}")

# List all files and folders in the current directory
print("\nContents of current directory:")
contents = os.listdir(current_dir)                                          # os.listdir(path) -> returns a list of the files in the directory
for item in contents:
    print(f"  - {item}")

In [ ]:
# Set a new working directory
os.chdir(r"D:\MASE6401\data")                           # os.chdir(path) -> changes the current working directory to the specified path
                                                        # use the raw string format r"path" to avoid Unicode errors when navigating directories

# Now running the same lines as before ...
current_dir = os.getcwd()                                              
print(f"Current working directory: {current_dir}")

### Setting Up Our Folder Directory

We need to navigate to the MASE6401 folder that contains our organized structure. <br>
We need to use the module `path` from `os`, which is used for common pathname manipulations

In [ ]:
# Navigate to the parent directory (assuming we're in the notebooks folder)

# If cwd not the specified folder use:
#current_dir = "path/to/MASE6401/Notebooks/"

# This gets us from MASE6401/notebooks/ to MASE6401/
project_root = os.path.dirname(current_dir)                      # os.path.dirname(path) -> returns the directory component of a pathname
print(f"\nProject root directory: {project_root}")

# Define paths to our important folders using os.path.join
# This ensures compatibility across different operating systems
data_dir = os.path.join(project_root, 'data')                    # os.path.join(path, path, path*) -> returns the result of joining two (or more) paths.
mts_metals_dir = os.path.join(data_dir, 'MTS_metals')
mts_sg_dir = os.path.join(data_dir, 'MTS_SG')
results_dir = os.path.join(project_root, 'results')

print(f"\nData directory: {data_dir}")
print(f"MTS Metals directory: {mts_metals_dir}")
print(f"Results directory: {results_dir}")

### 2.2 Working with Test Data Files

Let's explore what files we have in our MTS_metals folder.

In [ ]:
# Check if the directory exists
if os.path.exists(mts_metals_dir):                                             # os.path.exists(path) -> Test whether a path exists.  Returns True/False
    print("MTS_metals directory exists")
    
    # List all files in the MTS_metals directory
    test_files = os.listdir(mts_metals_dir)
    print(f"\nFound {len(test_files)} files:")
    for file in test_files:
        print(f"  - {file}")
else:
    print("MTS_metals directory not found. Please check your folder structure.")

### 2.3 Extracting Material Information from Filenames

Normally, in data science, we encode information using filenames or follow a given name convention. <br>

The naming convention here is: `TestNumber_MaterialCode_Identifier.xlsx`

For example: `4_1018_Crodata1.xlsx`
- Test number: 4
- Material: 1018 Steel
- Identifier: Crodata1 - Refers to Crosshead Data

Let's write a code to extract the material name/code from any filename.

In [ ]:
# Using previously found files
example_file = test_files[0]

# Example filename
#example_file = "replace_with_file_name.xlsx"                           # Use this in case the file name was not found

# Method 1: Using string split
# Split by underscore and take the second element (index 1)
parts = example_file.split('_')                                         # split every "_" instance
test_number = parts[0]                                                  # Should be the ID number
material_code = parts[1]                                                # Should be the Material ID 

print(f"\nFilename: {example_file}")
print(f"  Test Number: {test_number}")
print(f"  Material Code: {material_code}")

### 2.4 Creating Results Directory (if it doesn't exist)

Before we save results, we need to make sure our results directory exists.

In [ ]:
# Create results directory if it doesn't exist
if not os.path.exists(results_dir):    # os.makedirs(path, exist_ok=False) -> makes a dir with the specified path, when it already exists, it is omitted
    os.makedirs(results_dir)        
    print(f"Created results directory: {results_dir}")
else:
    print(f"Results directory already exists: {results_dir}")

### os Module Practice Exercise

Write code to:
1. List all Excel files (*.xlsx) in the MTS_metals directory
2. Extract the material code from each filename
3. Create a list of unique materials tested

**Hint:** Use a list comprehension and the `set()` function

In [ ]:
# Your code here:

## 3 - Loading Data into Pandas

Now that we know how to navigate our file system, let's load some real tensile testing data!



### 3.1 Reading Excel Files with Pandas

Excel files often contain multiple sheets. Our MTS testing data has three sheets:
- Sheet1: Metadata or setup information
- Sheet2: **Raw test data** (Load, Displacement, Strain, Time) ← We'll use this!
- Sheet3: Additional information

We'll focus on Sheet2, which contains the time-series data from the tensile test.

In [ ]:
# Define the path to our example file
file_path = os.path.join(mts_metals_dir, test_files[0])
print(f"Example test file: {file_path}")

### Loading a Specific Sheet

The `pd.read_excel()` function is the one used to read **.xlsx** (excel) files:

- `sheet_name`: Which sheet to load (can be name or index, 0-based)
- `header`: Which row contains column names (default is 0)
- `index_col`: Which column to use as the row index (optional)

In [ ]:
# Load Sheet2 (index 1, since indexing starts at 0)
df = pd.read_excel(file_path, sheet_name=1, header = 1)

print("Data loaded successfully!")
print(f"Data type: {type(df)}")

### First Look at the Data

In [ ]:
# Display the first few rows
print("\nFirst 5 rows of data:")
print(df.head())                                                                        # df.head() -> displays the first 5 rows of the dataframe object 

# Display the last few rows to see how the test ended
print("\nLast 5 rows of data:")
print(df.tail())                                                                        # df.tail() -> displays the first 5 rows of the dataframe object 

### 3.2 DataFrame vs Series: Understanding Pandas Data Structures

Pandas has two main data structures:

### DataFrame: 2D Data Structure
- Think of it as a **table** or **spreadsheet**
- Has rows AND columns
- Can contain multiple data types (one per column)
- What we get when we load an Excel file

### Series: 1D Data Structure
- Think of it as a **single column** or **array with labels**
- Has only one dimension
- Contains only one data type
- What we get when we select a single column from a DataFrame

In [ ]:
# The entire loaded data is a DataFrame
print(f"Type of df: {type(df)}")
print(f"Shape of df: {df.shape} (rows, columns)")

### Selecting Data: When do we get a Series?

In [ ]:
# Single column selection with single brackets → Returns a Series
load_series = df['Load (N)']
print(f"Type of load_series: {type(load_series)}")
print(f"Shape of load_series: {load_series.shape}")
print("\nFirst 5 values:")
print(load_series.head())

### Selecting Data: When do we get a DataFrame?

In [ ]:
# Method 2: Single column with double brackets → Returns a DataFrame
load_df = df[['Load (N)']]
print(f"\nType of load_df: {type(load_df)}")
print(f"Shape of load_df: {load_df.shape} (rows, columns)")
print("\nFirst 5 values:")
print(load_df.head())

# Method 3: Multiple columns → Always returns a DataFrame
load_time_df = df[['Load (N)', 'Time (sec)']]
print(f"\nType of load_time_df: {type(load_time_df)}")
print(f"Shape of load_time_df: {load_time_df.shape}")
print("\nFirst 5 values:")
print(load_time_df.head())

**Series** and **DataFrames** have different methods and behaviors:

In [ ]:
# Series have a .name attribute
print(f"Series name: {load_series.name}")

# DataFrames have .columns attribute 
print(f"DataFrame columns: {load_df.columns.tolist()}")

# Both have .shape, but Series shows only one dimension
print(f"Series shape: {load_series.shape}")     # (n,)
print(f"DataFrame shape: {load_df.shape}")      # (n, 1)

### Quick Reference: Series vs DataFrame

| Feature | Series | DataFrame |
|---------|--------|-----------|
| Dimensions | 1D | 2D |
| Analogy | Single column/array | Entire table |
| Selection | `df['column']` | `df[['col1', 'col2']]` |
| Attributes | `.name` | `.columns` |
| Shape | `(n,)` | `(n, m)` |
| When to use | Single variable analysis | Multi-variable analysis |

**Rule of thumb:** If you're analyzing one variable, work with a Series. If you need multiple variables, keep it as a DataFrame.

### 3.3 Loading Multiple Sheets

Sometimes we need data from multiple sheets. Here's how to load them all at once:



In [ ]:
# Load all sheets into a dictionary
all_sheets = pd.read_excel(file_path, sheet_name=None)

print(f"Number of sheets loaded: {len(all_sheets)}")
print(f"Sheet names: {list(all_sheets.keys())}")

# Access individual sheets from the dictionary
sheet2_data = all_sheets['Sheet2']
print(f"\nSheet2 has {len(sheet2_data)} rows and {len(sheet2_data.columns)} columns")

### Practice Exercise 3.1

1. Load the data from Sheet3 of the same file
2. Check if it's a DataFrame or Series
3. Display the first 10 rows

In [ ]:
# Your code here:

### Practice Exercise 3.2

Using the df DataFrame we loaded earlier:
1. Extract the 'Extensometer (mm)' column as a Series
2. Extract both 'Load (N)' and 'Extensometer (mm)' as a DataFrame
3. Check the type and shape of both
4. Print the first 3 values of each

In [ ]:
# Your code here:

## 4 - Exploratory Data Analysis Methods

Now that we have our data loaded, let's learn how to explore and understand it systematically. 

- Verify data quality
- Identify anomalies or sensor errors
- Understand test behavior before calculating properties
- Make informed decisions about data processing

We'll continue working with the DataFrame `df` we loaded in Module 3.

### 4.1 Basic DataFrame Information Methods

These methods give us a high-level overview of our dataset.

In [ ]:
# Display basic information about the DataFrame

# Shape: dimensions of the data (rows, columns)
print(f"\nDataFrame shape: {df.shape}")
print(f" {df.shape[0]} data points (rows)")
print(f" {df.shape[1]} variables (columns)")

# Column names
print(f"\nColumn names:")
for col in df.columns:
    print(f"  - {col}")

# Data types of each column
print(f"\nData types:")
print(df.dtypes)

In [ ]:
# Comprehensive info about the DataFrame
df.info()

### 4.2 Statistical Summary

The `.describe()` method provides key statistics for numerical columns.

In [ ]:
print(df.describe())

In [ ]:
# We can also get individual statistics - using the load column
print(f"\nMean load: {df['Load (N)'].mean():.2f} N")
print(f"Maximum load: {df['Load (N)'].max():.2f} N")
print(f"Minimum load: {df['Load (N)'].min():.2f} N")
print(f"Standard deviation of load: {df['Load (N)'].std():.2f} N")

print(f"\nTest duration: {df['Time (sec)'].max():.2f} seconds")
print(f"Maximum extensometer reading: {df['Extensometer (mm)'].max():.4f} mm")

### 4.3 Data Selection and Indexing

Pandas offers powerful ways to select specific data. Let's explore the main methods.

In [ ]:
# Method 1: Selecting columns

# Single column (returns Series)
load_data = df['Load (N)']                           # Calling df["Load (N)"] returns the series object for the "Load (N)" column
print(f"\nLoad data (first 5 values):")
print(load_data.head())

# Multiple columns (returns DataFrame)
load_time = df[['Load (N)', 'Time (sec)']]           # Calling df[["Load (N)", "Time (sec)"]] returns the dataframe object for the two columns
print(f"\nLoad and Time data (first 5 rows):")       # When selecting multiple columns use a list like approach -> ["col1", "col2", etc]
print(load_time.head())

### 4.4 Row Selection with .loc[] and .iloc[]

- **`.iloc[]`**: Select by integer position (like array indexing)
- **`.loc[]`**: Select by label/index value

In [ ]:
# .iloc[] - position-based indexing
print("\nFirst row using .iloc[0]:")                                 # Calling a specific row number 
print(df.iloc[0])

print("\nRows 5-7 using .iloc[5:8]:")                                # You can call a specific subset of rows using the notation [start: finish ::steps]
print(df.iloc[5:8])

print("\nFirst 3 rows, first 2 columns using .iloc[0:3, 0:2]:")      # Note the 2D dimension notation, here [rows, columns] indices
print(df.iloc[0:3, 0:2])

In [ ]:
# .loc[] - label-based indexing (using only the index values)
print("\nRow with index label 0 using .loc[0]:")                     # Calling a specific row number
print(df.loc[0])

print("\nRows 5-7 using .loc[5:7]:")                                 # Calling a subset of rows
print(df.loc[5:7])

### 4.5 Boolean Indexing (Filtering)

We can also filter data based on conditions.

In [ ]:
# Filter: Only data where Load is positive (actual tensile loading)
positive_load = df[df['Load (N)'] > 0]
print(f"\nOriginal data: {len(df)} points")
print(f"Positive load only: {len(positive_load)} points")
print(f"Filtered out {len(df) - len(positive_load)} preload points")

print("\nFirst 5 rows of positive load data:")
print(positive_load.head())

In [ ]:
# Multiple conditions: Load > 1000 N AND Time > 10 seconds
high_load = df[(df['Load (N)'] > 1000) & (df['Time (sec)'] > 10)]
print(f"Data points with Load > 1000 N and Time > 10 s: {len(high_load)}")

In [ ]:
# Filter using .between() method
mid_test = df[df['Time (sec)'].between(20, 40)]
print(f"Data points between 20-40 seconds: {len(mid_test)}")

**Logical operators for combining conditions:**
- `&` : AND (both conditions must be true)
- `|` : OR (at least one condition must be true)
- `~` : NOT (inverts the condition)

**Important:** Always use parentheses around each condition!

### 4.6 Finding Extreme Values

In materials testing, we often need to find peak loads, failure points, etc.

In [ ]:
# Maximum and minimum values
max_load = df['Load (N)'].max()
min_load = df['Load (N)'].min()

print(f"Maximum load: {max_load:.2f} N")
print(f"Minimum load: {min_load:.2f} N")

In [ ]:
# Find the INDEX where maximum occurs
max_load_idx = df['Load (N)'].idxmax()               # df[column].idmax() -> returns the idx at which the max value happens, similar to np.max()
min_load_idx = df['Load (N)'].idxmin()               # df[column].idmin() -> returns the idx at which the min value happens, similar to np.min()

print(f"Maximum load occurs at index: {max_load_idx}")
print(f"Minimum load occurs at index: {min_load_idx}")

In [ ]:
# Get the entire row where maximum load occurred
max_load_row = df.loc[max_load_idx]                 # Using the previous row idx for max value, return the whole row using the .loc method
print("\nFull data at maximum load:")
print(max_load_row)

print(f"  \n Peak load of {max_load:.2f} N occurred at:")
print(f"  Time: {max_load_row['Time (sec)']:.2f} seconds")
print(f"  Extensometer: {max_load_row['Extensometer (mm)']:.4f} mm")
print(f"  Crosshead: {max_load_row['Crosshead (mm)']:.4f} mm")

### 4.7 Handling Missing Data

Real-world data often has missing values (sensor failures, data transmission errors, etc.)

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())

# Total missing values
total_missing = df.isnull().sum().sum()
print(f"\nTotal missing values in dataset: {total_missing}")

**Methods for handling missing data:**
- `df.dropna()` - Remove rows with any missing values
- `df.dropna(subset=['Load (N)'])` - Remove rows with missing values in specific columns
- `df.fillna(value)` - Replace missing values with a specific value
- `df.fillna(method='ffill')` - Forward fill (use previous valid value)
- `df.fillna(df.mean())` - Fill with column mean

**For materials testing:** Missing data in critical columns (Load, Extensometer) 
usually means that data point should be removed, as interpolation could give 
misleading mechanical properties.

### Practice Exercise for Module 4

Using the loaded DataFrame `df`:

1. Find the time at which the maximum load occurred
2. Calculate the average load during the last 10 seconds of the test
3. How many data points have an extensometer reading greater than 0.1 mm?
4. Create a new DataFrame containing only data where Load > 100 N. How many data points does the new DataFrame contain?
5. What is the median crosshead displacement?

In [ ]:
# Your code here for 1:

In [ ]:
# Your code here for 2:

In [ ]:
# Your code here for 3:

In [ ]:
# Your code here for 4:

In [ ]:
# Your code here for 5: